In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_extract, expr


spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d142d8f2-fb36-4edc-b915-b45548db8173;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5

In [2]:
import os


# ⬇️ Параметры подключения к PostgreSQL
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

shops_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shops") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

shop_tz_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shop_timezone") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

# Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_tz_df.createOrReplaceTempView("shop_timezone")

In [3]:

shop_tz_df = shop_tz_df.where(''' time_zone != "" and try_cast(plant as INT) IS NOT NULL''')

st_timezone = (
    shops_df.join(shop_tz_df, shops_df.st_id==shop_tz_df.plant, "left")
    .select('st_id', 'shop_name', 'time_zone')
    .withColumn(
        "time_zone",
        expr(r"""
            CASE
                WHEN time_zone IS NULL OR time_zone = '' THEN 3
                ELSE CAST(regexp_extract(time_zone, '(\\d+)$', 1) AS INT)
            END
        """)
        )
    .orderBy('st_id')
    )

st_timezone.show()


+-----+-----------+---------+
|st_id|  shop_name|time_zone|
+-----+-----------+---------+
|  842|      Lenta|        7|
|  843|     Magnit|        4|
|  844|       Spar|        3|
|  845|Pyaterochka|        5|
|  846|      Lenta|        3|
|  847|      Diksi|        3|
|  848|      Lenta|        8|
|  849|   FixPrice|        3|
|  850|     Magnit|        3|
|  851|      Lenta|        3|
+-----+-----------+---------+

